<a href="https://colab.research.google.com/github/Ruba-Alharbi/MentalQA_NER-Reproduction/blob/main/RP__(F)_Few_shot_58_example_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Load files

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#import pandas as pd

#qr_df = pd.read_excel('58_full_df.xlsx', sheet_name= 'S2')

In [ ]:
## Addition:
# Merge the 58-question Drug gold labels with the reproduced GPT-4o, LLaMA, and ALLaM outputs,
# applying light question-text normalization for reliable matching and saving a file compatible with the published evaluation code.
import pandas as pd
import re
import unicodedata

def question_key(text):
    text = unicodedata.normalize("NFKC", str(text)).casefold()
    text = text.translate(str.maketrans({
        "أ": "ا", "إ": "ا", "آ": "ا",
        "ى": "ي", "ة": "ه",
        "ؤ": "و", "ئ": "ي",
        "ـ": ""
    }))
    text = re.sub(r"[\u0617-\u061A\u064B-\u0652\u0670]", "", text)
    return re.sub(r"[\W_]+", "", text)

def build_merged(gold_path, gpt_path, llama_path, allam_path, out_path):
    # Load Drug gold labels from the published 58-question evaluation set
    gold = pd.read_excel(gold_path, sheet_name="S2")
    gold.columns = gold.columns.str.strip().str.lower()

    index_col = [c for c in gold.columns if c.startswith("unnamed")][0]

    # Keep only the row ID, question text, and human-annotated Drug labels
    gold = gold[[index_col, "question", "medication mentioned"]].copy()
    gold.columns = ["Unnamed: 0", "question", "Medication Mentioned"]

    # Load reproduced few-shot or zero-shot model outputs
    gpt4o = pd.read_excel(gpt_path)[["question", "GPT-4o Output"]].copy()
    gpt4o.columns = ["question", "GPT-4o Output"]

    llama = pd.read_excel(llama_path)[["question", "LLaMA Output"]].copy()
    llama.columns = ["question", "LLaMA Output"]

    allam = pd.read_excel(allam_path)[["question", "Allam Output"]].copy()
    allam.columns = ["question", "Allam Output"]


    # Create temporary normalized keys for matching questions across files
    gold["match_key"] = gold["question"].apply(question_key)
    gpt4o["match_key"] = gpt4o["question"].apply(question_key)
    llama["match_key"] = llama["question"].apply(question_key)
    allam["match_key"] = allam["question"].apply(question_key)

     # Prevent ambiguous matches after question normalization
    if gpt4o["match_key"].duplicated().any():
        raise ValueError("Duplicate normalized questions found in the GPT-4o output file.")

    if llama["match_key"].duplicated().any():
        raise ValueError("Duplicate normalized questions found in the LLaMA output file.")

    if allam["match_key"].duplicated().any():
        raise ValueError("Duplicate normalized questions found in the ALLaM output file.")

    # Merge reproduced model outputs with the human Drug gold labels
    merged = (
        gold
        .merge(gpt4o[["match_key", "GPT-4o Output"]], on="match_key", how="left")
        .merge(llama[["match_key", "LLaMA Output"]], on="match_key", how="left")
        .merge(allam[["match_key", "Allam Output"]], on="match_key", how="left")
        .drop(columns=["match_key"])
    )

    # Handle the one known question whose wording differs slightly across files
    anchor = "انا صيدلي"

    target = merged["question"].astype(str).str.contains(
        anchor, regex=False, na=False
    )

    if target.sum() != 1:
        raise RuntimeError("The target pharmacist question (انا صيدلي) was not found exactly once in the 58-question file.")

    def find_source_row(source_df, source_name):
        matches = source_df[
            source_df["question"].astype(str).str.contains(
                anchor, regex=False, na=False
            )
        ]

        if len(matches) != 1:
            raise RuntimeError(
                f"The target pharmacist question (انا صيدلي) was not found exactly once in the {source_name} file."
            )

        return matches.iloc[0]

    gpt_row = find_source_row(gpt4o, "GPT-4o")
    llama_row = find_source_row(llama, "LLaMA")
    allam_row = find_source_row(allam, "ALLaM")

    # Insert the verified model outputs for the unmatched question
    merged.loc[target, "GPT-4o Output"] = gpt_row["GPT-4o Output"]
    merged.loc[target, "LLaMA Output"] = llama_row["LLaMA Output"]
    merged.loc[target, "Allam Output"] = allam_row["Allam Output"]

    # Verify that all 58 questions have a prediction from every model
    output_cols = ["GPT-4o Output", "LLaMA Output", "Allam Output"]

    for col in output_cols:
        missing = merged[col].isna().sum()
        print(f"{col}: missing {missing} / {len(merged)}")

    if merged[output_cols].isna().any().any():
        raise RuntimeError("Missing model outputs remain. Do not run the evaluation yet.")

    # Save the merged file use the sheet name expected by the published evaluation notebook
    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        merged.to_excel(writer, sheet_name="S2", index=False)

    print(f"Created {out_path}: {merged.shape}")  # Expected shape: (58, 6)
    return merged

# Create the 58-question evaluation file for reproduced few-shot outputs
merged_fewshot = build_merged(
    "58_full_df.xlsx",
    "few-gpt4o-llama_df.xlsx",
    "few-gpt4o-llama_df.xlsx",
    "few-allam_df.xlsx",
    "58_merged_fewshot.xlsx"
)

# Create the 58-question evaluation file for reproduced zero-shot outputs
merged_zeroshot = build_merged(
    "58_full_df.xlsx",
    "zero-gpt4o-llama_df.xlsx",
    "zero-gpt4o-llama_df.xlsx",
    "zero-allam_df.xlsx",
    "58_merged_zeroshot.xlsx"
)

In [ ]:
qr_df = pd.read_excel("58_merged_fewshot.xlsx", sheet_name="S2")
#For zero-shot
#qr_df = pd.read_excel("58_merged_zeroshot.xlsx", sheet_name="S2")

In [ ]:
# Normalize column names in both DataFrames
qr_df.columns = qr_df.columns.str.strip().str.lower()

# Confirm columns now
print(qr_df.columns)

In [ ]:
qr_df=qr_df.head(58)

In [ ]:
qr_df.head(2)

In [ ]:
# Drop the unwanted 'unnamed: 0' column
qr_df = qr_df.drop(columns=["unnamed: 0"])

# Check result
qr_df.head()

## Extract Drug Entity


In [ ]:
import re
from sklearn.metrics import precision_score, recall_score, f1_score

# Load the final merged file
df = qr_df.copy()

In [ ]:
# ====== Fresh cleaning pipeline (explicit columns; no auto-detect) ======
import re, pandas as pd

# ---------- Splitters / patterns ----------
# split on commas/Arabic commas/semicolons and common Arabic connectors (and/or/from/to/then)
_SEP = re.compile(r'[,\u060C;؛]|\s+(?:و|او|أو|من|عن|الى|إلى|ثم)\s+', re.IGNORECASE)

_DRUG_FIELD  = re.compile(r'"Drug"\s*:\s*(?:"([^"]*)"|\[(.*?)\])', re.IGNORECASE | re.DOTALL)
_PLAIN_FIELD = re.compile(r'\bDrug\s*[:：]\s*([^\n\r]+)', re.IGNORECASE)

# Numbers (EN + Arabic-Indic) and units
_NUM   = r'[\d\u0660-\u0669]'
_UNITS = r'(?:mg|ml|mcg|ug|g|iu|ملجم|مجم|مل|قرص|أقراص|اقراص|حبة|حبه|قطرة|جرعة|أمبول|امبول|حقنة|حقن)'

# Arabic definite article (with clitics), e.g., والـ، بالـ، للـ …
_AR_ARTICLE_RE = re.compile(r'^(?:[وفبكل]\s*)?(?:ال|أل|إل|آل|لل)ـ?\s*', re.IGNORECASE)

# Leading prepositions inside tokens (e.g., "من انديرال" -> "انديرال")
_AR_PREP_PREFIX_RE = re.compile(r'^(?:من|عن|الى|إلى|على|مع|بدون|بلا)\s+', re.IGNORECASE)

# Generic dosage-form / filler words (removed anywhere)
_GENERIC_WORDS = (
    r'اسم(?:ه|ها)?|غير\s*معروف|دواء|الدواء|ادوية|الأدوية|دوا|'
    r'برشام|'
    r'قرص|أقراص|اقراص|حبة|حبه|حبوب|'
    r'كبسول(?:ة|ات)?|'
    r'شراب|'
    r'مرهم|كريم|جل|محلول|لاصقة|'
    r'بخاخ|بخّاخ|'
    r'قطرة|قطرات|'
    r'حقنة|حقن|أمبول|امبول|امبولات|'
    r'مهدئ|مسكن|منوم|'
    r'مضاد(?:\s+حيوي)?'
)
_GENERIC_INLINE_RE = re.compile(rf'\b(?:{_GENERIC_WORDS})\b', re.IGNORECASE)

# Tokens to drop completely if they remain alone
_DROP_TOKENS = {
    "و","او","أو","من","عن","على","الى","إلى","ثم","في","مع","بدون","بلا",
    "اسم","اسمه","اسمها","الدواء","دواء","دوا","برشام","حبة","حبوب",
    "قرص","اقراص","أقراص","حقنة","حقن","امبول","أمبول","شراب",
    "مرهم","كريم","جل","بخاخ","قطرة","قطرات","لبوس","تحاميل","تحميلة",
    "غير","معروف"
}
_HAS_DIGIT_RE = re.compile(r'[\d\u0660-\u0669]')

In [ ]:
def _strip_noise(s: str) -> str:
    """Remove numbers/units and filler words anywhere in the token."""
    # number+unit and bare numbers anywhere (no word boundaries)
    s = re.sub(fr'{_NUM}+(?:[.,]{_NUM}+)?\s*{_UNITS}', ' ', s, flags=re.IGNORECASE)
    s = re.sub(fr'{_NUM}+(?:[.,]{_NUM}+)?', ' ', s)
    # generic/filler words
    s = _GENERIC_INLINE_RE.sub(' ', s)
    return s

In [ ]:
def clean_drug_name(name: str) -> str:
    """Normalize one token until only a plausible drug name remains."""
    s = str(name).strip().lower()

    # drop article and leading prepositions inside the token
    s = _AR_ARTICLE_RE.sub('', s)
    s = _AR_PREP_PREFIX_RE.sub('', s)

    # remove inline noise (numbers+units, dosage forms, 'اسمه/اسمها', 'دواء غير معروف', etc.)
    s = _strip_noise(s)

    # keep only letters/digits/-/+; squeeze spaces and trim
    s = re.sub(r'[^\w\-\+]+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip(' -+ ').strip()

    # drop if empty, still has digits, is a stop token, or is a single letter
    if (not s) or _HAS_DIGIT_RE.search(s) or (s in _DROP_TOKENS) or len(s) == 1:
        return ""
    return s

In [ ]:
def extract_drugs_from_model(text) -> list[str]:
    """Pull Drug values from JSON-like or plain text, clean, and dedupe."""
    if not isinstance(text, str) or not text.strip():
        return []
    drugs = []

    # JSON-like: "Drug": "..."  or  "Drug": ["a","b"]
    for m in _DRUG_FIELD.finditer(text):
        payload = (m.group(1) if m.group(1) is not None else m.group(2)) or ""
        payload = payload.replace('"', '').replace("'", '')
        for part in _SEP.split(payload):
            part = clean_drug_name(part)
            if part:
                drugs.append(part)

    # Fallback: plain "Drug: a, b"
    for m in _PLAIN_FIELD.finditer(text):
        payload = m.group(1)
        for part in _SEP.split(payload):
            part = clean_drug_name(part)
            if part:
                drugs.append(part)

    # dedupe preserving order
    seen, out = set(), []
    for d in drugs:
        if d and d not in seen:
            seen.add(d); out.append(d)
    return out

In [ ]:
def normalize_annotated(val) -> list[str]:
    """Normalize the ground-truth column to a list of clean names + dedupe."""
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return []
    if isinstance(val, list):
        parts = val
    else:
        text = str(val).strip().strip('[]').replace('"', '').replace("'", '')
        parts = [p for p in _SEP.split(text) if p.strip()]

    seen, out = set(), []
    for p in parts:
        c = clean_drug_name(p)
        if c and c not in seen:
            seen.add(c); out.append(c)
    return out

In [ ]:
def apply_cleaning(df: pd.DataFrame, *, med_col: str, model_cols: list[str], joiner: str = ", "):
    """Cleans in-place and returns df with strings (no lists). Requires explicit columns."""
    # drop 'unnamed' style columns (from index in CSV/Excel)
    df.drop(columns=[c for c in df.columns if c.lower().startswith("unnamed")],
            inplace=True, errors="ignore")

    # sanity checks
    missing = [c for c in [med_col, *model_cols] if c not in df.columns]
    if missing:
        raise ValueError(f"Missing expected columns: {missing}")

    # normalize annotated -> join
    df[med_col] = df[med_col].apply(normalize_annotated).apply(lambda xs: joiner.join(xs))

    # extract from each model output -> join
    for col in model_cols:
        df[col] = df[col].apply(extract_drugs_from_model).apply(lambda xs: joiner.join(xs))

    # move medication next to question if present
    if "question" in df.columns:
        cols = list(df.columns)
        cols.insert(cols.index("question")+1, cols.pop(cols.index(med_col)))
        df = df[cols]

    return df

In [ ]:
MED_COL    = "medication mentioned"
MODEL_COLS = ["gpt-4o output", "llama output", "allam output"]

In [ ]:
df = apply_cleaning(df, med_col=MED_COL, model_cols=MODEL_COLS, joiner=", ")
display(df.head(58))

## Evaluation

In [ ]:
# 1) Splitter: comma, Arabic comma "،", semicolons, and the Arabic "و" when used as "and"
_SEP = re.compile(r'[,\u060C;؛]|\s+و\s+', re.IGNORECASE)

def autodetect_cols(df: pd.DataFrame):
    """Find ground-truth and model columns by name (case-insensitive)."""
    truth_col = None
    for c in df.columns:
        lc = c.strip().lower()
        if "medication" in lc and "mention" in lc:
            truth_col = c
            break
    if truth_col is None:
        raise ValueError("Couldn't find the 'medication mentioned' column.")

    model_cols = [c for c in df.columns if "output" in c.lower()]
    if not model_cols:
        raise ValueError("Couldn't find any model output columns (names containing 'output').")
    return truth_col, model_cols

def to_set(val) -> set:
    """Turn a cell into a set of tokens with only splitting+trimming (no cleaning)."""
    if isinstance(val, list):
        parts = val
    else:
        text = "" if (val is None or (isinstance(val, float) and pd.isna(val))) else str(val)
        parts = _SEP.split(text)
    return {p.strip() for p in parts if str(p).strip()}

def evaluate(df: pd.DataFrame, truth_col=None, model_cols=None):
    """Micro-averaged P/R/F1 for each model with no additional normalization."""
    if truth_col is None or model_cols is None:
        truth_col, model_cols = autodetect_cols(df)

    y_true_sets = df[truth_col].apply(to_set)

    rows = []
    for col in model_cols:
        y_pred_sets = df[col].apply(to_set)

        TP = FP = FN = 0
        for t, p in zip(y_true_sets, y_pred_sets):
            TP += len(t & p)
            FP += len(p - t)
            FN += len(t - p)

        precision = TP / (TP + FP) if (TP + FP) else 0.0
        recall    = TP / (TP + FN) if (TP + FN) else 0.0
        f1        = 2*precision*recall/(precision+recall) if (precision+recall) else 0.0

        name = col.replace(" output", "").replace("Output", "").strip()
        print(f"{name}:\n  Precision: {precision:.2f}\n  Recall:    {recall:.2f}\n  F1 Score:  {f1:.2f}\n")
        rows.append((name, f1, precision, recall))

    summary = pd.DataFrame(rows, columns=["Model", "F1 Score", "Precision", "Recall"])
    summary = summary.round(2)

    return summary


summary = evaluate(df)
display(summary)

# Error Analysis
Error analysis using same cleaned values for evaluation

## Few-Shot

In [ ]:
# Create item-level Drug error records for the few-shot evaluation
# using the same cleaned values used by the published F1 evaluation.

import re
import pandas as pd

condition = "few_shot"

gold_column = "medication mentioned"

model_columns = {
    "GPT-4o": "gpt-4o output",
    "LLaMA": "llama output",
    "ALLaM": "allam output",
}

splitter = re.compile(r"[,،;؛]|\s+و\s+")

def to_drug_set(value):
    text = "" if pd.isna(value) else str(value)
    return {item.strip() for item in splitter.split(text) if item.strip()}

records = []

for row_id, row in df.iterrows():
    gold = to_drug_set(row[gold_column])

    for model_name, column_name in model_columns.items():
        prediction = to_drug_set(row[column_name])

        false_negatives = gold - prediction
        false_positives = prediction - gold

        if not false_negatives and not false_positives:
            error_shape = "exact"
        elif false_negatives and false_positives:
            error_shape = "mixed"
        elif false_negatives:
            error_shape = "false_negative_only"
        else:
            error_shape = "false_positive_only"

        records.append({
            "condition": condition,
            "row_id": row_id,
            "model": model_name,
            "question": row["question"],
            "gold_drugs": ", ".join(sorted(gold)),
            "predicted_drugs": ", ".join(sorted(prediction)),
            "false_negatives": ", ".join(sorted(false_negatives)),
            "false_positives": ", ".join(sorted(false_positives)),
            "gold_mention_count": len(gold),
            "prediction_mention_count": len(prediction),
            "input_type": "multiple_drugs" if len(gold) > 1 else "single_drug",
            "error_shape": error_shape,
            "is_error": error_shape != "exact",
        })

all_items = pd.DataFrame(records)
errors_only = all_items[all_items["is_error"]].copy()

by_input_type = (
    all_items
    .groupby(["model", "input_type"], as_index=False)
    .agg(
        items=("row_id", "count"),
        error_records=("is_error", "sum"),
        error_rate=("is_error", "mean"),
        false_negative_mentions=(
            "false_negatives",
            lambda values: sum(len(to_drug_set(value)) for value in values)
        ),
        false_positive_mentions=(
            "false_positives",
            lambda values: sum(len(to_drug_set(value)) for value in values)
        ),
    )
)

by_error_shape = (
    all_items
    .groupby(["model", "error_shape"], as_index=False)
    .size()
    .rename(columns={"size": "records"})
)

output_file = "few_shot_drug_error_analysis.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    all_items.to_excel(writer, sheet_name="all_items", index=False)
    errors_only.to_excel(writer, sheet_name="errors_only", index=False)
    by_input_type.to_excel(writer, sheet_name="by_input_type", index=False)
    by_error_shape.to_excel(writer, sheet_name="by_error_shape", index=False)

print(f"Saved {output_file}")

display(by_input_type)
display(by_error_shape)

## Zero-Shot

In [ ]:
# Create item-level Drug error records for the zero-shot evaluation
# using the same cleaned values used by the published F1 evaluation.

import re
import pandas as pd

condition = "zero_shot"

gold_column = "medication mentioned"

model_columns = {
    "GPT-4o": "gpt-4o output",
    "LLaMA": "llama output",
    "ALLaM": "allam output",
}

splitter = re.compile(r"[,،;؛]|\s+و\s+")

def to_drug_set(value):
    text = "" if pd.isna(value) else str(value)
    return {item.strip() for item in splitter.split(text) if item.strip()}

records = []

for row_id, row in df.iterrows():
    gold = to_drug_set(row[gold_column])

    for model_name, column_name in model_columns.items():
        prediction = to_drug_set(row[column_name])

        false_negatives = gold - prediction
        false_positives = prediction - gold

        if not false_negatives and not false_positives:
            error_shape = "exact"
        elif false_negatives and false_positives:
            error_shape = "mixed"
        elif false_negatives:
            error_shape = "false_negative_only"
        else:
            error_shape = "false_positive_only"

        records.append({
            "condition": condition,
            "row_id": row_id,
            "model": model_name,
            "question": row["question"],
            "gold_drugs": ", ".join(sorted(gold)),
            "predicted_drugs": ", ".join(sorted(prediction)),
            "false_negatives": ", ".join(sorted(false_negatives)),
            "false_positives": ", ".join(sorted(false_positives)),
            "gold_mention_count": len(gold),
            "prediction_mention_count": len(prediction),
            "input_type": "multiple_drugs" if len(gold) > 1 else "single_drug",
            "error_shape": error_shape,
            "is_error": error_shape != "exact",
        })

all_items = pd.DataFrame(records)
errors_only = all_items[all_items["is_error"]].copy()

by_input_type = (
    all_items
    .groupby(["model", "input_type"], as_index=False)
    .agg(
        items=("row_id", "count"),
        error_records=("is_error", "sum"),
        error_rate=("is_error", "mean"),
        false_negative_mentions=(
            "false_negatives",
            lambda values: sum(len(to_drug_set(value)) for value in values)
        ),
        false_positive_mentions=(
            "false_positives",
            lambda values: sum(len(to_drug_set(value)) for value in values)
        ),
    )
)

by_error_shape = (
    all_items
    .groupby(["model", "error_shape"], as_index=False)
    .size()
    .rename(columns={"size": "records"})
)

output_file = "zero_shot_drug_error_analysis.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    all_items.to_excel(writer, sheet_name="all_items", index=False)
    errors_only.to_excel(writer, sheet_name="errors_only", index=False)
    by_input_type.to_excel(writer, sheet_name="by_input_type", index=False)
    by_error_shape.to_excel(writer, sheet_name="by_error_shape", index=False)

print(f"Saved {output_file}")

display(by_input_type)
display(by_error_shape)

In [ ]:
# Pivot the error shape counts for side-by-side comparison across models
comparison_df = by_error_shape.pivot(index='error_shape', columns='model', values='records').fillna(0).astype(int)

# Order error shapes for readability (exact matches first, then errors)
order = ['exact', 'false_negative_only', 'false_positive_only', 'mixed']
comparison_df = comparison_df.reindex([o for o in order if o in comparison_df.index])

display(comparison_df)